# Creo un unico file in cui a partire dal dataset originale di trai e test concateno i risultati di performance dei singoli modelli

In [1]:
# ============================================================
# BLOCCO 2b — Arricchimento dataset (solo fold_id e predicted_label)
# ============================================================
import pandas as pd
import numpy as np
import ast

# ── Configurazione ──────────────────────────────────────────
DATASET_PATH = 'Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion.csv'
LABEL_COL    = 'label'

MODEL_FILES = {
    'bert':    'Results/Classification/bert_enriched.csv',
    'roberta': 'Results/Classification/roberta_enriched.csv',
    'llama':   'Results/Classification/llama_enriched.csv',
    'mistral': 'Results/Classification/mistral/24B_results_enriched.csv',
}

TRUE_LABEL_CANDIDATES = ['True_Labels',      'Val_True_Labels']
PRED_LABEL_CANDIDATES = ['Predicted_Labels', 'Val_Predictions']

# ── Helper ──────────────────────────────────────────────────

def _get_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"Nessuna delle colonne {candidates} trovata. Disponibili: {df.columns.tolist()}")


def _explode_fold(row, true_col, pred_col):
    ids         = ast.literal_eval(row['Article_IDs'])
    true_labels = ast.literal_eval(row[true_col])
    pred_labels = ast.literal_eval(row[pred_col])
    fold_id     = int(row['Fold'])

    return pd.DataFrame({
        'article_id':       ids,
        '_true_label':      true_labels,
        '_predicted_label': pred_labels,
        '_fold_id':         fold_id,
    })


# ── Pipeline principale ─────────────────────────────────────

print("Caricamento dataset originale...")
dataset = pd.read_csv(DATASET_PATH)
dataset = dataset.reset_index(drop=True)
dataset.index.name = 'article_id'
dataset = dataset.reset_index()
print(f"  Shape: {dataset.shape}")

for model_name, results_path in MODEL_FILES.items():
    print(f"\n{'─'*55}")
    print(f"Modello: {model_name.upper()}")

    results_df = pd.read_csv(results_path)
    true_col   = _get_col(results_df, TRUE_LABEL_CANDIDATES)
    pred_col   = _get_col(results_df, PRED_LABEL_CANDIDATES)

    model_df = pd.concat(
        [_explode_fold(row, true_col, pred_col) for _, row in results_df.iterrows()],
        ignore_index=True
    )

    # ── Check integrità label via article_id ────────────────
    check      = dataset[['article_id', LABEL_COL]].merge(
        model_df[['article_id', '_true_label']], on='article_id', how='inner'
    )
    mismatches = check[check[LABEL_COL] != check['_true_label']]
    if len(mismatches) == 0:
        print(f"  ✓ Integrità label OK — {len(check)} articoli verificati")
    else:
        print(f"  ✗ {len(mismatches)} disallineamenti label:")
        print(mismatches.head(10).to_string())
        raise ValueError(f"[{model_name}] True labels non corrispondono al dataset.")

    # ── Rename e join ────────────────────────────────────────
    model_df = model_df.rename(columns={
        '_predicted_label': f'{model_name}_predicted_label',
        '_fold_id':         f'{model_name}_fold_id',
    }).drop(columns=['_true_label'])   # già presente nel dataset come 'label'

    dataset = dataset.merge(model_df, on='article_id', how='left')

    for col in [f'{model_name}_predicted_label', f'{model_name}_fold_id']:
        dataset[col] = dataset[col].astype('Int64')

    print(f"  Colonne aggiunte: {[c for c in dataset.columns if c.startswith(model_name)]}")

# ── Salvataggio ──────────────────────────────────────────────
OUTPUT_PATH = 'Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched.csv'
dataset.to_csv(OUTPUT_PATH, index=False)

print(f"\n✓ Salvato in: {OUTPUT_PATH}")
print(f"  Shape finale: {dataset.shape}")
dataset.head(3)

Caricamento dataset originale...
  Shape: (19476, 13)

───────────────────────────────────────────────────────
Modello: BERT
  ✓ Integrità label OK — 19476 articoli verificati
  Colonne aggiunte: ['bert_predicted_label', 'bert_fold_id']

───────────────────────────────────────────────────────
Modello: ROBERTA
  ✓ Integrità label OK — 19476 articoli verificati
  Colonne aggiunte: ['roberta_predicted_label', 'roberta_fold_id']

───────────────────────────────────────────────────────
Modello: LLAMA
  ✓ Integrità label OK — 19476 articoli verificati
  Colonne aggiunte: ['llama_predicted_label', 'llama_fold_id']

───────────────────────────────────────────────────────
Modello: MISTRAL
  ✓ Integrità label OK — 19476 articoli verificati
  Colonne aggiunte: ['mistral_predicted_label', 'mistral_fold_id']

✓ Salvato in: Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched.csv
  Shape finale: (19476, 21)


,article_id,url,title,text,authors,domain,score,country,language,topics,...,opinion_advocacy,label,bert_predicted_label,bert_fold_id,roberta_predicted_label,roberta_fold_id,llama_predicted_label,llama_fold_id,mistral_predicted_label,mistral_fold_id
0,0,https://nemontananews.com/stories/670532029-1-...,1 gas station in cities throughout Phillips Co...,Federal Energy Regulatory Commission (FERC) Ch...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,...,Yes,0,0,5,0,5,0,5,0,5
1,1,https://nemontananews.com/stories/670175501-mc...,McCone County: Income Maintenance dependency r...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,...,Yes,0,0,5,0,5,0,5,0,5
2,2,https://nemontananews.com/stories/670489322-da...,"Daniels County residents received $13,488 per ...","John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,...,Yes,0,0,5,0,5,0,5,0,5


# Arricchisco il dataset con le confidenze dei modelli

## Bert, Roberta 

In [2]:
# ============================================================
# BLOCCO 3 (v2) — Inferenza con confidenze per classe
#                 BERT e RoBERTa, con check a tolleranza
# ============================================================
import ast, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm

# ── Configurazione ──────────────────────────────────────────
ENRICHED_PATH = 'Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched.csv'
OUTPUT_PATH   = 'Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched_with_probs.csv'
MODELS_BASE   = "Results/Classification/NON ELIMINARE Modelli salvati/Models"
RESULTS_BASE  = "Results/Classification"
LABEL_COL     = 'label'
BATCH_SIZE    = 32
MAX_LENGTH    = 256
N_SPLITS      = 5
TOLERANCE     = 0.005  # max 0.5% discrepanze accettabili (float32 CPU vs GPU mixed precision)

MODEL_CONFIGS = {
    'bert': {
        'hf_name':     'bert-base-uncased',
        'file_tmpl':   'bert_fold_{fold}.pt',
        'results_csv': f'{RESULTS_BASE}/bert_enriched.csv',
        'pred_col':    'bert_predicted_label',
        'true_col':    'True_Labels',
        'ids_col':     'Article_IDs',
    },
    'roberta': {
        'hf_name':     'roberta-large',
        'file_tmpl':   'roberta_fold_{fold}.pt',
        'results_csv': f'{RESULTS_BASE}/roberta_enriched.csv',
        'pred_col':    'roberta_predicted_label',
        'true_col':    'Val_True_Labels',
        'ids_col':     'Article_IDs',
    },
}

# ── Caricamento dataset ─────────────────────────────────────
print("Caricamento dataset enriched...")
dataset = pd.read_csv(ENRICHED_PATH)
print(f"  Shape: {dataset.shape}")
label_values = sorted(dataset[LABEL_COL].unique())
num_labels   = len(label_values)
print(f"  Classi: {label_values}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"  Device: {device}")

# Inizializza colonne confidenza
for model_name in MODEL_CONFIGS:
    for cls in label_values:
        dataset[f'{model_name}_prob_class_{cls}'] = np.nan

# ── Helper ──────────────────────────────────────────────────

def get_fold_article_ids(results_csv, fold_num, ids_col):
    df = pd.read_csv(results_csv)
    row = df[df['Fold'] == fold_num].iloc[0]
    return ast.literal_eval(row[ids_col])


def run_inference_fold(hf_name, model_path, article_ids, dataset,
                       num_labels, batch_size, max_length, device):
    """Ritorna (all_probs, article_ids_ordered) con ordine garantito."""
    subset = dataset.set_index('article_id').loc[article_ids].reset_index()
    texts  = subset['text'].fillna('').tolist()

    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    encodings = tokenizer(
        texts, truncation=True, padding='max_length',
        max_length=max_length, return_tensors='pt',
    )
    loader = DataLoader(
        TensorDataset(encodings.input_ids, encodings.attention_mask),
        batch_size=batch_size, shuffle=False,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        hf_name, num_labels=num_labels
    )
    model.load_state_dict(torch.load(model_path, map_location='cpu'))
    model.to(device)
    model.eval()

    all_probs = []
    with torch.no_grad():
        for batch in tqdm(loader, desc=f"  Inference {model_path.split('/')[-1]}", leave=False):
            out   = model(input_ids=batch[0].to(device),
                          attention_mask=batch[1].to(device))
            probs = F.softmax(out.logits, dim=-1).cpu().float().numpy()
            all_probs.append(probs)

    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return np.vstack(all_probs)   # (N, num_labels)


def check_predictions(model_name, fold_num, all_probs, article_ids,
                       saved_preds, label_values, tolerance):
    """
    Confronta argmax(probs) con le predicted_label salvate.
    Tollera piccole discrepanze da float32 CPU vs GPU mixed precision,
    stampando i dettagli per ogni caso border-line.
    Ritorna True se il check passa (con o senza tolleranza).
    """
    preds         = np.argmax(all_probs, axis=1)
    mismatch_mask = preds != saved_preds.astype(int)
    mismatches    = int(mismatch_mask.sum())
    mismatch_rate = mismatches / len(preds)

    if mismatches == 0:
        print(f"  ✓ Check argmax == predicted_label: OK ({len(preds)} articoli)")
        return True

    if mismatch_rate <= tolerance:
        print(f"  ⚠️  {mismatches}/{len(preds)} discrepanze ({mismatch_rate*100:.3f}%) "
              f"— entro tolleranza ({tolerance*100}%). "
              f"Causa probabile: float32 CPU vs mixed precision GPU.")
        bad_positions = np.where(mismatch_mask)[0]
        for pos in bad_positions[:10]:
            cls_now   = int(preds[pos])
            cls_saved = int(saved_preds[pos])
            prob_now   = all_probs[pos, cls_now]
            prob_saved = all_probs[pos, cls_saved]
            diff       = abs(prob_now - prob_saved)
            print(f"    article_id={article_ids[pos]} | "
                  f"argmax_now={cls_now} (p={prob_now:.6f}) | "
                  f"saved={cls_saved} (p={prob_saved:.6f}) | "
                  f"Δp={diff:.6f}")
        return True  # accettato con warning

    # Supera la soglia → errore reale
    raise ValueError(
        f"[{model_name}] Fold {fold_num}: {mismatches} discrepanze "
        f"({mismatch_rate*100:.2f}%) — supera la soglia {tolerance*100}%. "
        f"Verifica modello e ordine articoli."
    )


# ── Pipeline principale ──────────────────────────────────────
for model_name, cfg in MODEL_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"Modello: {model_name.upper()}")

    for fold_num in range(1, N_SPLITS + 1):
        print(f"\n  Fold {fold_num}/{N_SPLITS}")

        model_path  = (f"{MODELS_BASE}/Fold {fold_num}/"
                       f"{cfg['file_tmpl'].format(fold=fold_num)}")
        article_ids = get_fold_article_ids(cfg['results_csv'], fold_num, cfg['ids_col'])

        all_probs = run_inference_fold(
            hf_name     = cfg['hf_name'],
            model_path  = model_path,
            article_ids = article_ids,
            dataset     = dataset,
            num_labels  = num_labels,
            batch_size  = BATCH_SIZE,
            max_length  = MAX_LENGTH,
            device      = device,
        )

        # Predicted labels salvate nel dataset enriched
        saved_preds = (
            dataset.set_index('article_id')
            .loc[article_ids, cfg['pred_col']]
            .values
        )

        check_predictions(
            model_name  = model_name,
            fold_num    = fold_num,
            all_probs   = all_probs,
            article_ids = article_ids,
            saved_preds = saved_preds,
            label_values = label_values,
            tolerance   = TOLERANCE,
        )

        max_probs = all_probs.max(axis=1)
        print(f"  Confidenza media: {max_probs.mean():.4f} "
              f"(min={max_probs.min():.4f}, max={max_probs.max():.4f})")

        # ── Scrittura confidenze nel dataset ─────────────────
        for col_pos, art_id in enumerate(article_ids):
            row_pos = dataset.index[dataset['article_id'] == art_id][0]
            for cls_idx, cls_val in enumerate(label_values):
                dataset.at[row_pos, f'{model_name}_prob_class_{cls_val}'] = \
                    float(all_probs[col_pos, cls_idx])

    print(f"\n  Colonne aggiunte: "
          f"{[f'{model_name}_prob_class_{c}' for c in label_values]}")

# ── Salvataggio ──────────────────────────────────────────────
dataset.to_csv(OUTPUT_PATH, index=False)
print(f"\n✓ File salvato: {OUTPUT_PATH}")
print(f"  Shape finale: {dataset.shape}")
dataset.head(3)

/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/cuda/__init__.py:628: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


Caricamento dataset enriched...
  Shape: (19476, 21)
  Classi: [0, 1, 2, 3, 4]
  Device: cuda

Modello: BERT

  Fold 1/5


2026-06-29 13:27:24.584982: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-29 13:27:24.602032: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782739644.620789 1731386 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782739644.626265 1731386 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782739644.641005 1731386 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

  ✓ Check argmax == predicted_label: OK (3895 articoli)
  Confidenza media: 0.8803 (min=0.2846, max=0.9998)

  Fold 2/5


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  Inference bert_fold_2.pt:   0%|                       | 0/127 [00:00<?, ?it/s]/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1520: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


...

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  Inference roberta_fold_4.pt:   0%|                    | 0/121 [00:00<?, ?it/s]/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1520: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


  ⚠️  2/3868 discrepanze (0.052%) — entro tolleranza (0.5%). Causa probabile: float32 CPU vs mixed precision GPU.
    article_id=5145 | argmax_now=2 (p=0.388962) | saved=3 (p=0.387088) | Δp=0.001874
    article_id=12823 | argmax_now=0 (p=0.379324) | saved=1 (p=0.378810) | Δp=0.000515
  Confidenza media: 0.9300 (min=0.3225, max=0.9999)

  Fold 5/5


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
  Inference roberta_fold_5.pt:   0%|                    | 0/126 [00:00<?, ?it/s]/home/fabio.pinelli/anaconda3/lib/python3.11/site-packages/torch/nn/modules/module.py:1520: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `RobertaSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


  ⚠️  1/4001 discrepanze (0.025%) — entro tolleranza (0.5%). Causa probabile: float32 CPU vs mixed precision GPU.
    article_id=13625 | argmax_now=2 (p=0.453740) | saved=1 (p=0.453196) | Δp=0.000544
  Confidenza media: 0.9354 (min=0.2405, max=1.0000)

  Colonne aggiunte: ['roberta_prob_class_0', 'roberta_prob_class_1', 'roberta_prob_class_2', 'roberta_prob_class_3', 'roberta_prob_class_4']

✓ File salvato: Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched_with_probs.csv
  Shape finale: (19476, 31)


,article_id,url,title,text,authors,domain,score,country,language,topics,...,bert_prob_class_0,bert_prob_class_1,bert_prob_class_2,bert_prob_class_3,bert_prob_class_4,roberta_prob_class_0,roberta_prob_class_1,roberta_prob_class_2,roberta_prob_class_3,roberta_prob_class_4
0,0,https://nemontananews.com/stories/670532029-1-...,1 gas station in cities throughout Phillips Co...,Federal Energy Regulatory Commission (FERC) Ch...,Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,...,0.999750,0.000091,0.000037,0.000038,0.000084,0.999949,0.000014,0.000006,0.000006,0.000024
1,1,https://nemontananews.com/stories/670175501-mc...,McCone County: Income Maintenance dependency r...,"John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,...,0.999751,0.000097,0.000033,0.000044,0.000074,0.999950,0.000016,0.000007,0.000006,0.000021
2,2,https://nemontananews.com/stories/670489322-da...,"Daniels County residents received $13,488 per ...","John Lettieri, President and CEO of Economic I...",Northeast Montana News,nemontananews.com,32.0,US,en,Political news or commentary,...,0.999749,0.000094,0.000030,0.000047,0.000080,0.999951,0.000015,0.000006,0.000006,0.000022


In [2]:
from huggingface_hub import login
login()

## LLama

In [1]:
!ls

'0 - NgScraperPol.ipynb'
'1 - Data preparation.ipynb'
'2 - Reliability - 5Fold - Separate.ipynb'
'3 - Reliability - 5Fold - Separate - mistral22B.ipynb'
'4 - Plotter.ipynb'
'5 - Political Error Analysis.ipynb'
'6 - Domain inpact analisys.ipynb'
'7 - Get prediction scores.ipynb'
 Dataset
 Results
 ScraperLogs


In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
import ast, gc
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

print("GPU:", torch.cuda.get_device_name(0))
print("bf16 support:", torch.cuda.is_bf16_supported()) # su V100S deve essere False

# --- PATH ---
BASE = ""
INPUT_PATH = "Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched_with_probs.csv"
OUTPUT_PATH = "Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched_with_probs_LLAMA_float16.csv"
MODELS_BASE = "Results/Classification/NON ELIMINARE Modelli salvati/Models"
RESULTS_CSV = "Results/Classification/llama_enriched.csv"
MODEL_ID = "meta-llama/Meta-Llama-3-8B"

LABEL_COL = 'label'
BATCH_SIZE = 32 # V100S 32GB regge bene
MAX_LENGTH = 256

LLAMA_FILE_MAP = {
    1: "llama_fold_1.pt",
    2: "llama_fold_2.pt",
    3: "llama_fold_3.pt",
    4: "llama_fold_4.pt",
    5: "llama_fold_5.pt",
}

dataset = pd.read_csv(INPUT_PATH)
label_values = sorted(dataset[LABEL_COL].unique())

# crea colonne se mancano
for cls in label_values:
    col = f'llama_prob_class_{cls}'
    if col not in dataset.columns:
        dataset[col] = np.nan
if 'llama_predicted_label' not in dataset.columns:
    dataset['llama_predicted_label'] = -1

lora_config = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","down_proj","up_proj"],
    lora_dropout=0.05, bias="none", task_type="SEQ_CLS",
)

# --- FLOAT16 per V100 ---
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

results_df = pd.read_csv(RESULTS_CSV)
def get_ids(f): return ast.literal_eval(results_df[results_df['Fold']==f].iloc[0]['Article_IDs'])

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.getenv("HF_TOKEN"))
tokenizer.pad_token = tokenizer.eos_token

for fold_num in range(1,6):
    print(f"\n{'='*60}\nFold {fold_num}")
    model_path = f"{MODELS_BASE}/Fold {fold_num}/{LLAMA_FILE_MAP[fold_num]}"
    article_ids = get_ids(fold_num)

    subset = dataset.set_index('article_id').loc[article_ids].reset_index()
    enc = tokenizer(subset['text'].fillna('').tolist(), truncation=True, padding=True,
                    max_length=MAX_LENGTH, return_tensors='pt')
    loader = DataLoader(TensorDataset(enc.input_ids, enc.attention_mask), batch_size=BATCH_SIZE)

    print("Carico modello 4-bit diretto in GPU...")
    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        quantization_config=quantization_config,
        num_labels=len(label_values),
        torch_dtype=torch.float16,
        device_map=None,           # niente accelerate, niente CPU
        low_cpu_mem_usage=False,   # niente meta-device
        token=os.getenv("HF_TOKEN")
    )
    base_model.config.pad_token_id = tokenizer.eos_token_id
    base_model.config.use_cache = False



    model = prepare_model_for_kbit_training(base_model)
    model = get_peft_model(model, lora_config)
    state_dict = torch.load(model_path, map_location='cpu')
    model.load_state_dict(state_dict, strict=False)
    model.eval()

    all_probs = []
    with torch.inference_mode(), torch.amp.autocast('cuda', dtype=torch.float16):
        for batch in tqdm(loader, desc="Inferenza"):
            logits = model(batch[0].to(0), attention_mask=batch[1].to(0)).logits
            all_probs.append(F.softmax(logits, dim=-1).float().cpu().numpy())

    all_probs = np.vstack(all_probs)
    preds = all_probs.argmax(1)

    # --- SOVRASCRIVI SEMPRE ---
    for i, aid in enumerate(article_ids):
        row = dataset.index[dataset['article_id']==aid][0]
        dataset.at[row, 'llama_predicted_label'] = int(preds[i])
        for j, cls in enumerate(label_values):
            dataset.at[row, f'llama_prob_class_{cls}'] = float(all_probs[i,j])

    print(f"Fold {fold_num} completato - {len(preds)} predizioni aggiornate")

    del model, base_model, loader, enc
    torch.cuda.empty_cache()
    gc.collect()

dataset.to_csv(OUTPUT_PATH, index=False)
print("✓ Salvato:", OUTPUT_PATH)

In [2]:
import torch
print(torch.cuda.get_device_name(0))
print("compute capability:", torch.cuda.get_device_capability(0))
print("bf16 support:", torch.cuda.is_bf16_supported())

Tesla V100S-PCIE-32GB
compute capability: (7, 0)
bf16 support: False


In [3]:
import gc
import torch

# Manually remove large objects from previous run
llama_objects = ['model', 'tokenizer', 'train_loader', 'test_loader', 
                'train_dataset', 'test_dataset', 'train_encodings',
                'test_encodings', 'optimizer', 'scaler']

for obj in llama_objects:
    if obj in globals():
        del globals()[obj]

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared: {} MB".format(torch.cuda.memory_allocated() // 1024**2))

GPU memory cleared: 8 MB


## Mistral

In [4]:
import gc
import torch

# Manually remove large objects from previous run
llama_objects = ['model', 'tokenizer', 'train_loader', 'test_loader', 
                'train_dataset', 'test_dataset', 'train_encodings',
                'test_encodings', 'optimizer', 'scaler']

for obj in llama_objects:
    if obj in globals():
        del globals()[obj]

# Force garbage collection
gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared: {} MB".format(torch.cuda.memory_allocated() // 1024**2))

GPU memory cleared: 8 MB


In [ ]:
import os
import ast
import gc
import logging
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding
from peft import PeftModel
from tqdm import tqdm


# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────

INPUT_PATH   = 'Dataset/CleannedDataset/newDataframeWithOldCleanedPoliticsVersion_enriched_with_probs.csv'

BASE_OUTPUT  = "Results/Classification/mistral/24B_results_enriched"
OUTPUT_CSV   = BASE_OUTPUT + "_checkpoint.csv"
LOG_FILE     = BASE_OUTPUT + ".log"

MODELS_BASE  = "Results/Classification/mistral/Models"
RESULTS_CSV  = "Results/Classification/mistral/24B_results_enriched.csv"

MODEL_ID     = "mistralai/Mistral-Small-24B-Base-2501"

BATCH_SIZE   = 4
MAX_LENGTH   = 1024
N_SPLITS     = 5


# ─────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────

logging.basicConfig(
    filename=LOG_FILE,
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

def log(msg):
    print(msg)
    logging.info(msg)

log("=== START RUN ===")


# ─────────────────────────────────────────────
# DATASET
# ─────────────────────────────────────────────

dataset = pd.read_csv(INPUT_PATH)

label_values = sorted(dataset["label"].unique())
num_labels = len(label_values)

log(f"Dataset shape: {dataset.shape}")
log(f"Labels: {label_values}")

for cls in label_values:
    col = f"mistral_prob_class_{cls}"
    if col not in dataset.columns:
        dataset[col] = np.nan


# ─────────────────────────────────────────────
# LOAD FOLDS
# ─────────────────────────────────────────────

results_df = pd.read_csv(RESULTS_CSV)

def get_fold_ids(fold):
    row = results_df[results_df["Fold"] == fold].iloc[0]
    return ast.literal_eval(row["Article_IDs"])


# ─────────────────────────────────────────────
# TOKENIZER (🔥 FIX PAD TOKEN)
# ─────────────────────────────────────────────

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# IMPORTANTISSIMO FIX
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.pad_token_id = tokenizer.eos_token_id


collator = DataCollatorWithPadding(tokenizer)


# ─────────────────────────────────────────────
# DATASET (FIXED BATCH SAFE)
# ─────────────────────────────────────────────

class TextDataset(Dataset):
    def __init__(self, tokenizer, texts):
        self.texts = texts
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=MAX_LENGTH,
            padding=False
        )


# ─────────────────────────────────────────────
# RESUME CHECK
# ─────────────────────────────────────────────

def fold_done(article_ids):
    col = f"mistral_prob_class_{label_values[0]}"
    sample = article_ids[0]
    v = dataset.loc[dataset["article_id"] == sample, col].values
    return len(v) > 0 and not pd.isna(v[0])


# ─────────────────────────────────────────────
# LOOP FOLDS
# ─────────────────────────────────────────────

for fold in range(1, N_SPLITS + 1):

    log(f"================ FOLD {fold} ================")

    adapter_path = f"{MODELS_BASE}/fold_{fold}"
    article_ids = get_fold_ids(fold)

    if fold_done(article_ids):
        log(f"Fold {fold} already done → SKIP")
        continue

    subset = dataset.set_index("article_id").loc[article_ids].reset_index()
    texts = subset["text"].fillna("").tolist()

    log(f"Fold {fold} samples: {len(texts)}")

    ds = TextDataset(tokenizer, texts)

    loader = DataLoader(
        ds,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        collate_fn=collator
    )


    # ─────────────────────────────────────────
    # MODEL LOAD
    # ─────────────────────────────────────────

    log("Loading model...")

    base_model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=num_labels,
        torch_dtype=torch.float16,
        device_map="auto"
    )

    base_model.config.pad_token_id = tokenizer.pad_token_id

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    log("Model loaded ✔")


    # ─────────────────────────────────────────
    # INFERENCE SAFE
    # ─────────────────────────────────────────

    all_probs = []

    for i, batch in enumerate(tqdm(loader, desc=f"Fold {fold}")):

        try:
            with torch.no_grad():
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"]
                )

                probs = F.softmax(outputs.logits, dim=-1)

            all_probs.append(probs.cpu().numpy())

        except Exception as e:
            log(f"Batch {i} error: {e}")
            continue

        if i % 50 == 0:
            log(f"Checkpoint fold {fold} batch {i}")


    all_probs = np.vstack(all_probs)

    log(f"Fold {fold} completed: {len(all_probs)} samples")


    # ─────────────────────────────────────────
    # SAVE RESULTS
    # ─────────────────────────────────────────

    for i, art_id in enumerate(article_ids[:len(all_probs)]):

        row_idx = dataset.index[dataset["article_id"] == art_id][0]

        for c_idx, c_val in enumerate(label_values):
            dataset.at[row_idx, f"mistral_prob_class_{c_val}"] = float(all_probs[i, c_idx])


    dataset.to_csv(OUTPUT_CSV, index=False)
    log(f"Saved checkpoint after fold {fold}")


    # cleanup
    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()


# ─────────────────────────────────────────────
# FINAL SAVE
# ─────────────────────────────────────────────

dataset.to_csv(OUTPUT_CSV, index=False)

log("=== DONE ===")

print("✔ OUTPUT:", OUTPUT_CSV)
print("✔ LOG:", LOG_FILE)